#### Import Necessary Packages

In [1]:
import json
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor

workspace = notebookutils.runtime.context.get("currentWorkspaceName")

StatementMeta(, 0740cbc3-d34e-49f6-b887-1fc94e3f274c, 3, Finished, Available, Finished)

#### Define required parameters

In [7]:
# # Parameters for manual run (A new cell will be created below this cell during the Runtime with the parameters from the pipeline)
# table_metadata = """
# [
#     {
#         "object_id": "1",
#         "source_modified_date_column": "ModifyDate",
#         "source_created_date_column": "CreateDate",
#         "sink_name": "lh_raw",
#         "sink_schema_name": "etrac_keters", 
#         "sink_object_name": "application_object_type"
#     },
#     {
#         "object_id": "22",
#         "source_modified_date_column": "ModifyDate",
#         "source_created_date_column": "CreateDate",
#         "sink_name": "lh_raw",
#         "sink_schema_name": "etrac_keters", 
#         "sink_object_name": "vendor"

#     },
#     {
#         "object_id": "17", 
#         "source_modified_date_column": "ModifyDate",
#         "source_created_date_column": "CreateDate",
#         "sink_name": "lh_raw",
#         "sink_schema_name": "etrac_keters", 
#         "sink_object_name": "repair_order"
#     }
# ]
# """

StatementMeta(, 0740cbc3-d34e-49f6-b887-1fc94e3f274c, 9, Finished, Available, Finished)

#### Check Table Presence and Retrieve Latest Modified and Created Dates When Available

In [8]:
# Parse input data
input_data = json.loads(table_metadata)
results_dict = {}

def process_table(item):
    object_id = item['object_id']
    sink_name = item['sink_name']
    sink_schema_name = item['sink_schema_name']
    sink_object_name = item['sink_object_name']
    source_modified_date_column = item.get('source_modified_date_column')
    source_created_date_column = item.get('source_created_date_column')

    # Initialize result
    result = {
        "table_exists": False,
        "row_count": 0,
        "last_modified_date": '1900-01-01 00:00:00.000000',
        "last_created_date": '1900-01-01 00:00:00.000000'
    }

    table_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{sink_name}.Lakehouse/Tables/{sink_schema_name}/{sink_object_name}"

    try:
        if not DeltaTable.isDeltaTable(spark, table_path):
            return object_id, result

        result["table_exists"] = True
        
        # Build query based on available columns
        columns_to_select = ["COUNT(*) AS row_count"]
        
        if source_modified_date_column:
            columns_to_select.append(f"MAX({source_modified_date_column}) AS max_modified_date")
        if source_created_date_column:
            columns_to_select.append(f"MAX({source_created_date_column}) AS max_created_date")

        query = f"SELECT {', '.join(columns_to_select)} FROM {sink_name}.{sink_schema_name}.{sink_object_name}"
        
        # Single query execution
        query_result = spark.sql(query).collect()[0]
        
        result["row_count"] = query_result['row_count']
        
        if source_modified_date_column and query_result['max_modified_date']:
            result["last_modified_date"] = str(query_result['max_modified_date'])
        if source_created_date_column and query_result['max_created_date']:
            result["last_created_date"] = str(query_result['max_created_date'])

    except Exception as e:
        print(f"Error processing {sink_schema_name}.{sink_object_name}: {str(e)}")

    return object_id, result

# Process tables in parallel
with ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(process_table, input_data)
    results_dict = dict(results)

# Output results
output_json = json.dumps(results_dict)

StatementMeta(, 0740cbc3-d34e-49f6-b887-1fc94e3f274c, 10, Finished, Available, Finished)

#### Return Output to Pipeline

In [ ]:
# Return the results to pipeline
output = {
    "status": "success", 
    "total_tables_processed": len(output_json),
    "tables_metadata": output_json
}

# Return to pipeline
mssparkutils.notebook.exit(json.dumps(output))

### Stop Session

In [10]:
mssparkutils.session.stop()

StatementMeta(, 0740cbc3-d34e-49f6-b887-1fc94e3f274c, 12, Finished, Available, Finished)